# Phishguard-Transformer - Step 2: Classic ML Baseline (XGBoost)

Building the classic ML side of the project here - hand-picked features from the URL, fed into XGBoost. This is the baseline I'll compare DistilBERT against later, so using the exact same train/val/test split from Step 1.

### Load Step 1's splits

Reusing the same CSVs from Step 1, not touching the original dataset again.

In [1]:
import pandas as pd
import numpy as np
import re
from urllib.parse import urlparse

train = pd.read_csv("train.csv")
val = pd.read_csv("val.csv")
test = pd.read_csv("test.csv")

print("Train:", train.shape, "Val:", val.shape, "Test:", test.shape)
train.head()

Train: (8400, 2) Val: (1800, 2) Test: (1800, 2)


,URL,label
0,https://www.plasticfreejuly.org,0
1,https://provi78arge.webcindario.com/,1
2,https://www.roseyleebooks.com,0
3,https://att-106905.weeblysite.com/,1
4,https://tinyurl.com/blocca-pagamento,1


### Feature extraction

XGBoost needs numbers, not raw text, so pulling out 12 features from each URL that are known phishing signals:
- url_length, num_dots, num_hyphens, num_digits, num_special_chars - basic structural stuff, phishing URLs tend to be longer/messier
- has_https - legit sites are usually https, though this is getting weaker as a signal since phishing sites use https too now
- has_ip - using a raw IP instead of a domain is a classic red flag
- num_subdirs, domain_length - path/domain structure
- has_at_symbol - can be used to trick browsers about the real destination
- suspicious_word_count - counts words like login/verify/secure/account etc
- entropy - how "random" the characters look, auto-generated malicious domains tend to score higher here

In [2]:
def extract_features(url):
    url = str(url)
    parsed = urlparse(url if '://' in url else 'http://' + url)
    domain = parsed.netloc

    length = len(url)
    num_dots = url.count('.')
    num_hyphens = url.count('-')
    num_digits = sum(c.isdigit() for c in url)
    num_special = len(re.findall(r'[^a-zA-Z0-9.\-/:]', url))
    has_https = int(parsed.scheme == 'https')
    has_ip = int(bool(re.match(r'^(\d{1,3}\.){3}\d{1,3}$', domain)))
    num_subdirs = url.count('/')
    domain_length = len(domain)
    at_symbol = int('@' in url)
    suspicious_words = ['login','verify','update','secure','account','bank','confirm','signin','webscr']
    suspicious_count = sum(w in url.lower() for w in suspicious_words)

    # Shannon entropy: measures how "random" the character distribution is
    probs = [url.count(c)/length for c in set(url)] if length > 0 else [0]
    entropy = -sum(p*np.log2(p) for p in probs if p > 0)

    return pd.Series({
        'url_length': length, 'num_dots': num_dots, 'num_hyphens': num_hyphens,
        'num_digits': num_digits, 'num_special_chars': num_special, 'has_https': has_https,
        'has_ip': has_ip, 'num_subdirs': num_subdirs, 'domain_length': domain_length,
        'has_at_symbol': at_symbol, 'suspicious_word_count': suspicious_count, 'entropy': entropy
    })

# quick test on one example URL
extract_features("https://provi78arge.webcindario.com/")

url_length               36.000000
num_dots                  2.000000
num_hyphens               0.000000
num_digits                2.000000
num_special_chars         0.000000
has_https                 1.000000
has_ip                    0.000000
num_subdirs               3.000000
domain_length            27.000000
has_at_symbol             0.000000
suspicious_word_count     0.000000
entropy                   4.308271
dtype: float64

Ran it on one phishing example to check it works - looks fine, has_ip is 0 (correct, it's a domain not an IP), values look sane.

### Apply to all three splits

In [3]:
feat_train = train['URL'].apply(extract_features)
feat_val = val['URL'].apply(extract_features)
feat_test = test['URL'].apply(extract_features)

print("Feature matrix shape (train):", feat_train.shape)
feat_train.describe()

Feature matrix shape (train): (8400, 12)


,url_length,num_dots,num_hyphens,num_digits,num_special_chars,has_https,has_ip,num_subdirs,domain_length,has_at_symbol,suspicious_word_count,entropy
count,8400.000000,8400.000000,8400.000000,8400.000000,8400.000000,8400.000000,8400.00000,8400.000000,8400.000000,8400.000000,8400.000000,8400.000000
mean,37.295119,2.264643,0.418690,2.383929,0.324405,0.743929,0.00250,2.515595,21.907619,0.007024,0.041667,3.977515
std,58.368660,1.340242,2.908504,23.348203,4.920526,0.436488,0.04994,1.377275,9.654764,0.083518,0.235906,0.327114
min,15.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.00000,2.000000,4.000000,0.000000,0.000000,2.623429
25%,25.000000,2.000000,0.000000,0.000000,0.000000,0.000000,0.00000,2.000000,16.000000,0.000000,0.000000,3.772055
50%,29.000000,2.000000,0.000000,0.000000,0.000000,1.000000,0.00000,2.000000,20.000000,0.000000,0.000000,3.940555
75%,36.000000,2.000000,0.000000,0.000000,0.000000,1.000000,0.00000,3.000000,25.000000,0.000000,0.000000,4.116300
max,4247.000000,90.000000,250.000000,2011.000000,390.000000,1.000000,1.00000,68.000000,105.000000,1.000000,3.000000,5.335261


Checked the distributions with describe() - nothing looks broken, suspicious_word_count is mostly 0 which makes sense since most URLs won't contain those words.

### Train XGBoost

Going with XGBoost since it's the standard choice for this kind of tabular data - same reasoning as my churn project. Not using the validation set for tuning right now, keeping it simple.

In [4]:
from xgboost import XGBClassifier

model = XGBClassifier(
    n_estimators=200,
    max_depth=5,
    learning_rate=0.1,
    eval_metric='logloss',
    random_state=42
)

model.fit(feat_train, train['label'])
print("Model trained.")

Model trained.


### Evaluate on test set

Only looking at the test set here since it's never been touched during training. Tracking accuracy, precision, recall and F1 - not just accuracy, since precision tells me how many false alarms I'm generating and recall tells me how many actual phishing sites I'm missing.

In [5]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

pred_test = model.predict(feat_test)

acc = accuracy_score(test['label'], pred_test)
prec = precision_score(test['label'], pred_test)
rec = recall_score(test['label'], pred_test)
f1 = f1_score(test['label'], pred_test)

print(f"Accuracy:  {acc:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall:    {rec:.4f}")
print(f"F1 score:  {f1:.4f}")

print("\nFull classification report:")
print(classification_report(test['label'], pred_test, target_names=['legitimate (0)', 'phishing (1)']))

Accuracy:  0.9928
Precision: 0.9966
Recall:    0.9889
F1 score:  0.9927

Full classification report:
                precision    recall  f1-score   support

legitimate (0)       0.99      1.00      0.99       900
  phishing (1)       1.00      0.99      0.99       900

      accuracy                           0.99      1800
     macro avg       0.99      0.99      0.99      1800
  weighted avg       0.99      0.99      0.99      1800



### Confusion matrix

In [6]:
cm = confusion_matrix(test['label'], pred_test)
cm_df = pd.DataFrame(
    cm,
    index=['Actual: legitimate', 'Actual: phishing'],
    columns=['Predicted: legitimate', 'Predicted: phishing']
)
cm_df

,Predicted: legitimate,Predicted: phishing
Actual: legitimate,897,3
Actual: phishing,10,890


3 legit URLs wrongly flagged, 10 phishing URLs missed out of 900 each. The missed phishing ones are the more important error type since those are real attacks getting through - could look into these more later.

### Feature importance

Curious which of the 12 features the model is actually relying on.

In [7]:
importances = pd.Series(model.feature_importances_, index=feat_train.columns).sort_values(ascending=False)
importances

has_https                0.495668
num_subdirs              0.486802
num_dots                 0.005370
url_length               0.004893
num_digits               0.003450
num_hyphens              0.002402
entropy                  0.000852
domain_length            0.000563
has_ip                   0.000000
num_special_chars        0.000000
has_at_symbol            0.000000
suspicious_word_count    0.000000
dtype: float32

has_https and num_subdirs together make up ~98% of the importance - basically everything else barely mattered. Kind of surprising, I expected entropy/suspicious words to matter more. Worth mentioning this in interviews since it's a specific, real finding and not just "it worked."



### Save the model

In [8]:
import json

model.save_model("xgboost_phishing_model.json")

results = {
    "model": "XGBoost (classic ML, lexical features)",
    "accuracy": acc,
    "precision": prec,
    "recall": rec,
    "f1": f1
}
with open("xgboost_results.json", "w") as f:
    json.dump(results, f, indent=2)

print("Saved xgboost_phishing_model.json and xgboost_results.json")
print(results)

Saved xgboost_phishing_model.json and xgboost_results.json
{'model': 'XGBoost (classic ML, lexical features)', 'accuracy': 0.9927777777777778, 'precision': 0.9966405375139977, 'recall': 0.9888888888888889, 'f1': 0.992749581706637}


### Done

XGBoost baseline: 99.28% accuracy, 99.27% F1 on the real test set. Model and results saved so I can load them later without retraining. Next: fine-tune DistilBERT on raw URL text and compare.